# Algoritmos genéticos

Este notebook **es** el código del proyecto (sin HTML ni canvas).

1. **Ruta del repartidor** (permutaciones, minimizar distancia).
2. **Aterrizaje de nave** (comandos + física, maximizar fitness).

**Población N** = cuántos individuos hay. **No** es el número de barrios ni de genes.

## Ciclo común

```
crear población aleatoria
repetir cada generación:
    evaluar fitness de todos
    copiar los elite mejores (elitismo)
    mientras falten individuos:
        elegir padres (selección)
        cruzar con probabilidad Pc
        mutar con probabilidad Pm
    esa lista pasa a ser la nueva población
```

---
# 1. Ruta más corta (TSP)

**Problema:** visitar todos los barrios una vez y volver. Cromosoma = orden de visita (permutación).

- Fitness = `100000 / distancia` (más corta → más fitness).
- Selección = **ruleta** (más fitness = más chance).
- Cruce = **OX** (no repite ciudades). Pc = 0.85.
- Mutación = intercambiar 2 barrios. Pm = 0.12.
- Elitismo = 2.

In [ ]:
import random, math

random.seed(7)

CITIES = [(90, 80), (400, 120), (700, 90), (820, 300),
          (600, 420), (280, 400), (120, 260), (480, 250)]
C = len(CITIES)
N, PC, PM, ELITE, GENS = 40, 0.85, 0.12, 2, 80

def dist(i, j):
    a, b = CITIES[i], CITIES[j]
    return math.hypot(a[0] - b[0], a[1] - b[1])

def tour_len(order):
    return sum(dist(order[i], order[(i + 1) % C]) for i in range(C))

def make(order):
    km = tour_len(order)
    return {"order": order[:], "km": km, "fit": 100000 / km}

def shuffle_route():
    o = list(range(C))
    random.shuffle(o)
    return make(o)

def roulette(pop):
    total = sum(p["fit"] for p in pop)
    r = random.random() * total
    for p in pop:
        r -= p["fit"]
        if r <= 0:
            return p
    return pop[-1]

def ox(a, b):
    start = random.randrange(C)
    end = start + 1 + random.randrange(C - 1)
    slice_ = a[start:min(end, C)]
    rest = [x for x in b if x not in slice_]
    child, r = [], 0
    for i in range(C):
        if start <= i < min(end, C):
            child.append(slice_[i - start])
        else:
            child.append(rest[r]); r += 1
    return child

def mutate(order, pm):
    o = order[:]
    if random.random() < pm:
        i, j = random.randrange(C), random.randrange(C)
        if i == j:
            j = (j + 1) % C
        o[i], o[j] = o[j], o[i]
    return o

pop = [shuffle_route() for _ in range(N)]
pop.sort(key=lambda p: p["km"])
print("gen 0  mejor distancia =", round(pop[0]["km"], 1))

for gen in range(1, GENS + 1):
    nxt = [make(pop[i]["order"]) for i in range(ELITE)]
    while len(nxt) < N:
        p1, p2 = roulette(pop), roulette(pop)
        child = ox(p1["order"], p2["order"]) if random.random() < PC else p1["order"][:]
        nxt.append(make(mutate(child, PM)))
    pop = sorted(nxt, key=lambda p: p["km"])
    if gen in (1, 10, 20, 40, 80):
        print(f"gen {gen:3d}  mejor distancia = {pop[0]['km']:.1f}  orden = {pop[0]['order']}")

print("mejor final:", pop[0]["order"], "km =", round(pop[0]["km"], 1))

### Qué hace cada función (ruta)

| Función | Rol |
|---|---|
| `tour_len` | Distancia del ciclo (incluye volver). |
| `make` | Convierte un orden en individuo con `km` y `fit`. |
| `roulette` | Elige padre: más `fit` → más probabilidad. |
| `ox` | Cruce que **no repite** barrios. |
| `mutate` | Intercambia dos posiciones con probabilidad Pm. |

La distancia del mejor debería **bajar** y luego aplanarse: elitismo copia las 2 mejores rutas intactas.

---
# 2. Aterrizaje de nave

**Cromosoma:** 12 genes. Cada gen = `{rot: -1,0,1`, `thrust: 0 o 1}` y se aplica 8 ticks de física.

**Aterrizaje válido** (las tres a la vez): en la pista, despacio (`|vy|<5`, `|vx|<4`), derecha (`|ang|<0.85`).

- Fitness: acercarse en X, ir lento, ir derecho, +4000 si toca pista, **+10000 si aterriza**.
- Selección = torneo entre los mejores.
- Cruce = 1 punto. Pc = 0.86.
- Mutación = reescribir cada gen con Pm = 0.08.
- Elitismo = 5.

### Por qué después de un punto el camino ya no cambia

`simulate(genes)` es **determinista**: el mismo ADN produce el mismo vuelo.
Cuando alguien aterriza, su fitness es enorme (+10000). El elitismo **copia esos genes sin mutar**.
Nadie suele superar al campeón, así que `pop[0]` se clona: ves siempre la misma trayectoria. Eso es convergencia, no un error.

In [ ]:
import random, math

random.seed(3)

W, GROUND = 960, 455
PAD_X, PAD_W = 320, 320
STEPS, HOLD = 12, 8
GRAVITY, THRUST = 0.14, 0.40
N, PC, PM, ELITE, GENS = 60, 0.86, 0.08, 5, 25

def rand_gene():
    return {"rot": random.choice([-1, 0, 0, 1]), "thrust": 1 if random.random() < 0.4 else 0}

def verdict(x, vx, vy, ang, hit):
    on_pad = PAD_X <= x <= PAD_X + PAD_W
    slow = abs(vy) < 5 and abs(vx) < 4
    straight = abs(ang) < 0.85
    landed = hit and on_pad and slow and straight
    return on_pad, slow, straight, landed

def simulate(genes):
    x, y, vx, vy, ang = W / 2 + 70, 70, 0.0, 0.0, 0.0
    hit = False
    for g in genes:
        for _ in range(HOLD):
            ang = max(-1, min(1, ang + g["rot"] * 0.08))
            if g["thrust"]:
                vx += math.sin(ang) * THRUST
                vy -= math.cos(ang) * THRUST
            vy += GRAVITY
            vx *= 0.99
            x += vx; y += vy
            if y < 10:
                y = 10
                vy = max(vy, 0)
            x = min(max(x, 16), W - 16)
            if y >= GROUND - 14:
                y = GROUND - 14
                hit = True
                break
        if hit:
            break
    on_pad, slow, straight, landed = verdict(x, vx, vy, ang, hit)
    cx = PAD_X + PAD_W / 2
    fit = 4000 / (1 + abs(x - cx) * 0.15)
    fit += 1500 / (1 + abs(vy))
    fit += 600 / (1 + abs(vx))
    fit += 400 / (1 + abs(ang) * 2)
    fit += 1000 if hit else y * 1.5
    if on_pad and hit:
        fit += 4000
    if landed:
        fit += 10000
    return {"genes": [dict(g) for g in genes], "fit": fit, "landed": landed, "x": x, "vy": vy}

def eval_all(pop):
    out = [simulate(p["genes"]) for p in pop]
    out.sort(key=lambda s: s["fit"], reverse=True)
    return out

def tournament(pop):
    best = pop[random.randrange(min(12, len(pop)))]
    for _ in range(3):
        c = pop[random.randrange(min(18, len(pop)))]
        if c["fit"] > best["fit"]:
            best = c
    return best

def crossover(a, b, pc):
    if random.random() > pc:
        return [dict(g) for g in a]
    cut = 1 + random.randrange(STEPS - 1)
    return [dict(g) for g in a[:cut] + b[cut:]]

def mutate(genes, pm):
    return [rand_gene() if random.random() < pm else dict(g) for g in genes]

pop = eval_all([{"genes": [rand_gene() for _ in range(STEPS)]} for _ in range(N)])
print("gen 0  fitness =", round(pop[0]["fit"], 1), "aterrizó =", pop[0]["landed"])

first = None
same_as_prev = 0
prev_genes = None

for gen in range(1, GENS + 1):
    nxt = [{"genes": [dict(g) for g in pop[i]["genes"]]} for i in range(ELITE)]
    while len(nxt) < N:
        child = mutate(crossover(tournament(pop)["genes"], tournament(pop)["genes"], PC), PM)
        nxt.append({"genes": child})
    pop = eval_all(nxt)
    if pop[0]["landed"] and first is None:
        first = gen
    if prev_genes == pop[0]["genes"]:
        same_as_prev += 1
    else:
        same_as_prev = 0
    prev_genes = pop[0]["genes"]
    n_land = sum(1 for p in pop if p["landed"])
    cloned = pop[0]["genes"] == prev_genes
    print(f"gen {gen:3d}  fit={pop[0]['fit']:.0f}  aterrizó={pop[0]['landed']}  naves_ok={n_land}  mismo_ADN_campeón={cloned}")
    prev_genes = pop[0]["genes"]

print("primer aterrizaje en generación:", first)
print("el mejor se clona por elitismo; simulate(mismo ADN) = mismo vuelo")

### Qué hace cada función (nave)

| Función | Rol |
|---|---|
| `rand_gene` | Un comando aleatorio. |
| `simulate` | Física determinista: mismo ADN → mismo camino. |
| `verdict` | ¿Pista + despacio + derecha? |
| `eval_all` | Fitness de todos y ordena (mejor primero). |
| `tournament` | Elige un padre entre los mejores. |
| `crossover` | Corta el ADN en un punto. |
| `mutate` | Cada gen se puede reescribir. |

Cuando `aterrizó=True` y `naves_ok` se queda alto, el campeón ya no cambia: elitismo 5 copia el ADN ganador.

## Parámetros (para la exposición)

| Parámetro | Significado | Ruta | Nave |
|---|---|---|---|
| N población | Cuántas soluciones a la vez | 40 | 60 |
| Pc | Probabilidad de cruzar padres | 0.85 | 0.86 |
| Pm | Probabilidad de mutar | 0.12 (swap) | 0.08 (por gen) |
| Elitismo | Mejores que pasan intactos | 2 | 5 |

Sube el notebook a Colab: Archivo → Subir notebook, o arrastra `Algoritmos_Geneticos.ipynb`.